In [2]:
import os
import re
import random

In [ ]:
import pandas as pd

data = pd.read_csv('competition_data.csv')
submission = pd.read_csv('submission.csv')
submission_aux = pd.read_csv('submission.csv')

In [4]:
data

,Unnamed: 0,ts,username,platform,conn_country,user_agent_decrypted,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,reason_start,shuffle,TARGET
0,110163,2018-03-11T05:05:45Z,11145402699,"iOS 11.0 (iPhone8,1)",AR,NaN,Crazy For U,Big Time Rush,24/seven,spotify:track:3jFfr89lnSmb4QBtfG8JBP,clickrow,False,False
1,66026,2023-06-05T10:42:33Z,11145402699,ios,AR,unknown,Nada Personal - Remasterizado 2007,Soda Stereo,Me Verás Volver (Hits & Más),spotify:track:09TTeexnlKewZdjOak2sV2,trackdone,True,False
2,116790,2018-07-01T02:04:51Z,11145402699,"iOS 11.0 (iPhone8,1)",AR,unknown,Good Times,CHIC,Risqué,spotify:track:0G3fbPbE1vGeABDEZF0jeG,trackdone,True,True
3,18431,2019-09-08T04:58:07Z,11145402699,"iOS 12.4 (iPhone8,1)",AR,unknown,Verano del 92,Los Piojos,3er Arco,spotify:track:1NXvuBAq48QrxRFQZVmORQ,trackdone,False,True
4,82941,2017-07-13T18:13:52Z,11145402699,"iOS 11.0 (iPhone8,1)",AR,NaN,Simpatico,Ekko Park,Simpatico,spotify:track:2gYJY0sIx1ErgTIha2nPRg,trackdone,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
100139,78756,2017-04-06T20:31:14Z,11145402699,"iOS 10.2.1 (iPad6,8,1)",AR,NaN,Losing My Religion,R.E.M.,In Time: The Best Of R.E.M. 1988-2003,spotify:track:12axV6NUqaYH3yFUWwArzr,trackdone,True,False
100140,12585,2021-11-03T21:06:07Z,11145402699,"iOS 15.1 (iPhone12,3)",AR,unknown,Cerca De La Revolucion,Charly García,Piano Bar,spotify:track:66grIvFLGrI4tNhggO2DAd,trackdone,True,False
100141,93960,2022-01-14T01:13:42Z,11145402699,"iOS 15.2 (iPhone12,3)",AR,unknown,A Los Jóvenes De Ayer - Remastered 2012,Serú Girán,Bicicleta,spotify:track:6YAl320SfxLO4rVIZTFKqG,trackdone,True,False
100142,74339,2024-05-01T17:51:16Z,11145402699,ios,AR,NaN,heat not hot,Serengeti,heat not hot,spotify:track:0rKLO1hXpmoIthQgJKoczN,trackdone,True,False


In [5]:
#eliminamos datos particulares de Chona, son siempre los mismos, y los que consideramos irrelevanets
data = data.drop(columns=['username', 'conn_country', 'user_agent_decrypted'])
submission = submission.drop(columns=['username', 'conn_country', 'user_agent_decrypted'])

In [6]:
# del timestamp, solo nos interesa distinguir hora, el dia de la semana, el mes y el año

data['ts'] = pd.to_datetime(data['ts'])
submission['ts'] = pd.to_datetime(submission['ts'])

# Extraer la hora
data['hour'] = data['ts'].dt.hour
submission['hour'] = submission['ts'].dt.hour

# Extraer el día de la semana (0 es lunes, 6 es domingo)
data['day_of_week'] = data['ts'].dt.dayofweek
submission['day_of_week'] = submission['ts'].dt.dayofweek

# Extraer el mes
data['month'] = data['ts'].dt.month
submission['month'] = submission['ts'].dt.month

# Extraer el año
data['year'] = data['ts'].dt.year
submission['year'] = submission['ts'].dt.year

In [7]:
# del dispositivo, solo queremos saber si se reprodujo en celular (iphone) o no
data['is_iphone'] = data['platform'].apply(lambda x: 1 if ('ios' in x or 'iOS' in x) else 0)
submission['is_iphone'] = submission['platform'].apply(lambda x: 1 if ('ios' in x or 'iOS' in x) else 0)

# dropeamos platform
data = data.drop(columns=['platform'])
submission = submission.drop(columns=['platform'])

In [8]:
data

,Unnamed: 0,ts,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,reason_start,shuffle,TARGET,hour,day_of_week,month,year,is_iphone
0,110163,2018-03-11 05:05:45+00:00,Crazy For U,Big Time Rush,24/seven,spotify:track:3jFfr89lnSmb4QBtfG8JBP,clickrow,False,False,5,6,3,2018,1
1,66026,2023-06-05 10:42:33+00:00,Nada Personal - Remasterizado 2007,Soda Stereo,Me Verás Volver (Hits & Más),spotify:track:09TTeexnlKewZdjOak2sV2,trackdone,True,False,10,0,6,2023,1
2,116790,2018-07-01 02:04:51+00:00,Good Times,CHIC,Risqué,spotify:track:0G3fbPbE1vGeABDEZF0jeG,trackdone,True,True,2,6,7,2018,1
3,18431,2019-09-08 04:58:07+00:00,Verano del 92,Los Piojos,3er Arco,spotify:track:1NXvuBAq48QrxRFQZVmORQ,trackdone,False,True,4,6,9,2019,1
4,82941,2017-07-13 18:13:52+00:00,Simpatico,Ekko Park,Simpatico,spotify:track:2gYJY0sIx1ErgTIha2nPRg,trackdone,True,False,18,3,7,2017,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100139,78756,2017-04-06 20:31:14+00:00,Losing My Religion,R.E.M.,In Time: The Best Of R.E.M. 1988-2003,spotify:track:12axV6NUqaYH3yFUWwArzr,trackdone,True,False,20,3,4,2017,1
100140,12585,2021-11-03 21:06:07+00:00,Cerca De La Revolucion,Charly García,Piano Bar,spotify:track:66grIvFLGrI4tNhggO2DAd,trackdone,True,False,21,2,11,2021,1
100141,93960,2022-01-14 01:13:42+00:00,A Los Jóvenes De Ayer - Remastered 2012,Serú Girán,Bicicleta,spotify:track:6YAl320SfxLO4rVIZTFKqG,trackdone,True,False,1,4,1,2022,1
100142,74339,2024-05-01 17:51:16+00:00,heat not hot,Serengeti,heat not hot,spotify:track:0rKLO1hXpmoIthQgJKoczN,trackdone,True,False,17,2,5,2024,1


In [9]:
#antes de seguir con operaciones por columna, separamos validation y train
# la idea es que train tenga 80% de datos de cada año, y val el resto
train_parts = []
val_parts = []

for year, group in data.groupby('year'):
    group = group.sort_values('ts')  # mantener orden cronológico dentro del año
    split_idx = int(len(group) * 0.8)
    train_parts.append(group.iloc[:split_idx])
    val_parts.append(group.iloc[split_idx:])

train_data = pd.concat(train_parts).sort_values('ts').reset_index(drop=True)
val_data = pd.concat(val_parts).sort_values('ts').reset_index(drop=True)

In [10]:
train_data

,Unnamed: 0,ts,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,reason_start,shuffle,TARGET,hour,day_of_week,month,year,is_iphone
0,74917,2014-06-27 18:01:16+00:00,Algo Demencial,Los Tipitos,Push,spotify:track:0rqD0zvqI4GrvyUxyQ7Ij3,fwdbtn,True,True,18,4,6,2014,1
1,74918,2014-06-29 20:10:10+00:00,Iron Man - Live,Black Sabbath,Past Lives,spotify:track:1XtZ3GROvmy4JrT6uMvsD8,NaN,False,True,20,6,6,2014,1
2,74919,2014-06-29 20:10:54+00:00,Sultans Of Swing,Dire Straits,Dire Straits,spotify:track:3LTMnFa0hhwisyq6ILahyj,fwdbtn,False,True,20,6,6,2014,1
3,74920,2014-06-29 20:11:16+00:00,Fortunate Son,Creedence Clearwater Revival,Willy And The Poor Boys,spotify:track:7I2lPuuiOkpKtZlhr4zUpT,fwdbtn,False,True,20,6,6,2014,1
4,74921,2014-09-04 21:46:24+00:00,"Comptine d'un autre été, l'après-midi",Yann Tiersen,Amelie from Montmartre,spotify:track:2AkcjsKlRbIBYGAgpQVFii,NaN,False,True,21,3,9,2014,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80105,74211,2024-04-23 18:10:42+00:00,Redbone,Childish Gambino,"""Awaken, My Love!""",spotify:track:0WtDGnWL2KrMCk0mI1Gpwz,trackdone,True,False,18,1,4,2024,1
80106,74212,2024-04-23 18:11:29+00:00,Thunder,Imagine Dragons,Evolve,spotify:track:1zB4vmk8tFRmM9UULNzbLB,trackdone,True,True,18,1,4,2024,1
80107,74213,2024-04-23 18:11:31+00:00,Pump Up The Jam - Edit,Technotronic,Best Of,spotify:track:0UAEHlFR79k9CJvknSGUNf,fwdbtn,True,True,18,1,4,2024,1
80108,74214,2024-04-23 18:11:33+00:00,Young Folks,Peter Bjorn and John,Writer's Block,spotify:track:4dyx5SzxPPaD8xQIid5Wjj,fwdbtn,True,True,18,1,4,2024,1


In [11]:
submission = submission.sort_values('ts').reset_index(drop=True)

In [12]:
# La idea es primero, simplificar los motivos de inicio de la canción, y luego hacer One-Hot Encoding sobre la columna simplificada
# Definir una función para simplificar los motivos
def simplify_reason(reason):
    if reason == 'fwdbtn':
        return 'fwdbtn'
    elif reason == 'trackdone':
        return 'trackdone'
    else:
        return 'other'

# Aplicar la simplificación
train_data['reason_start'] = train_data['reason_start'].apply(simplify_reason)
val_data['reason_start'] = val_data['reason_start'].apply(simplify_reason)
submission['reason_start'] = submission['reason_start'].apply(simplify_reason)

In [13]:
# Ahora hacer One-Hot Encoding sobre la columna simplificada
train_encoded_reasons = pd.get_dummies(train_data['reason_start'], prefix='reason')
val_encoded_reasons = pd.get_dummies(val_data['reason_start'], prefix='reason')
submission_encoded_reasons = pd.get_dummies(submission['reason_start'], prefix='reason')

# Asegurar que ambas tengan las mismas columnas (por si en validación falta alguna categoría)
for col in ['reason_fwdbtn', 'reason_trackdone', 'reason_other']:
    if col not in train_encoded_reasons:
        train_encoded_reasons[col] = 0
    if col not in val_encoded_reasons:
        val_encoded_reasons[col] = 0
    if col not in submission_encoded_reasons:
        submission_encoded_reasons[col] = 0

# Asegurar el mismo orden de columnas
train_encoded_reasons = train_encoded_reasons[['reason_fwdbtn', 'reason_trackdone', 'reason_other']]
val_encoded_reasons = val_encoded_reasons[['reason_fwdbtn', 'reason_trackdone', 'reason_other']]
submission_encoded_reasons = submission_encoded_reasons[['reason_fwdbtn', 'reason_trackdone', 'reason_other']]

# convertir a entero los bools
for col in train_encoded_reasons.columns:
    train_encoded_reasons[col] = train_encoded_reasons[col].astype(int)
    val_encoded_reasons[col] = val_encoded_reasons[col].astype(int)
    submission_encoded_reasons[col] = submission_encoded_reasons[col].astype(int)
    

# Concatenar al DataFrame original
train_data = pd.concat([train_data.reset_index(drop=True), train_encoded_reasons.reset_index(drop=True)], axis=1)
val_data = pd.concat([val_data.reset_index(drop=True), val_encoded_reasons.reset_index(drop=True)], axis=1)
submission = pd.concat([submission.reset_index(drop=True), submission_encoded_reasons.reset_index(drop=True)], axis=1)

In [14]:
submission

,Unnamed: 0,ts,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,reason_start,shuffle,hour,day_of_week,month,year,is_iphone,reason_fwdbtn,reason_trackdone,reason_other
0,74916,2014-06-27 18:01:15+00:00,Mejor,Los Tipitos,Push,spotify:track:5LFl6vXC2CwcciAbymL4jZ,other,False,18,4,6,2014,1,0,0,1
1,74923,2014-09-04 21:46:57+00:00,"Circles - Based On Ludovico Einaudi ""Experience""",Ludovico Einaudi,In a Time Lapse,spotify:track:0mEsOEi4rWBy0IXE5oTKr2,fwdbtn,False,21,3,9,2014,1,1,0,0
2,74924,2014-09-04 21:48:51+00:00,Primavera,Ludovico Einaudi,Divenire,spotify:track:0fzw4BBD5FRJtPuQbUUKzJ,other,False,21,3,9,2014,1,0,0,1
3,74933,2016-06-23 21:07:59+00:00,NaN,NaN,NaN,NaN,other,False,21,3,6,2016,0,0,0,1
4,74934,2016-06-23 21:08:03+00:00,NaN,NaN,NaN,NaN,other,False,21,3,6,2016,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25032,74898,2024-05-22 15:28:48+00:00,Zafar,La Vela Puerca,A Contraluz,spotify:track:1wIUWGdTdhVk5gIPd0ULxX,trackdone,True,15,2,5,2024,1,0,1,0
25033,74899,2024-05-22 15:35:04+00:00,Un Loco En La Calesita,Juan Carlos Baglietto,Baglietto,spotify:track:3mHOEGxXbUpk5CZDgQhrUP,fwdbtn,True,15,2,5,2024,1,1,0,0
25034,74900,2024-05-22 15:39:44+00:00,Dulce condena - Edición Aniversario,Los Rodriguez,Sin Documentos,spotify:track:4Pk1N5mY14kO5N3JcADgb2,trackdone,True,15,2,5,2024,1,0,1,0
25035,74901,2024-05-22 15:39:49+00:00,Yo No Quiero Volverme Tan Loco,Charly García,Pubis Angelical / Yendo De La Cama Al Living,spotify:track:68LeIVjVDRMXPlfdFHhID6,trackdone,True,15,2,5,2024,1,0,1,0


In [15]:
#pip install spotipy
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import webbrowser

In [16]:
def get_spotify_auth():
    # Configura el objeto SpotifyOAuth
    auth_manager = SpotifyOAuth(
        client_id="bd28f888d5594351a74597b8c4750b07",
        client_secret="2ad3d9ea7c00425792697d028c5afbd5",
        redirect_uri="http://127.0.0.1:1234/",
        scope="user-top-read",
        open_browser=False
    )
    if auth_manager != None:
        return auth_manager
    else:
        # Si no se puede obtener el token, abre el navegador para la autenticación
        print("Error al obtener el token")

def chunk_list(lst, size):
    """Divide la lista 'lst' en sublistas de tamaño 'size'."""
    return [lst[i:i + size] for i in range(0, len(lst), size)]


In [17]:
def get_songs_durations(df):
    manager = get_spotify_auth()
    sp = spotipy.Spotify(auth_manager=manager)
    # Obtener la duración de las canciones
    # Obtener los URIs únicos y válidos
    tracks = df['spotify_track_uri'].dropna().unique()  # Elimina NaN directamente
    tracks = [uri for uri in tracks if isinstance(uri, str)]  # Filtra que sean strings

    # Dividir en chunks de hasta 50 elementos
    chunks = chunk_list(tracks, 50)
    uri_to_duration = {}
    for i, chunk in enumerate(chunks):
        print(i)
        try:
            tracks_info = sp.tracks(chunk)  # Consulta a la API
            for track in tracks_info.get('tracks', []):
                if track:
                    uri = track.get('uri')
                    duration = track.get('duration_ms')
                    uri_to_duration[uri] = duration
                else:
                    print("⚠️  Track no encontrado o no disponible.")
        except Exception as e:
            print(f"❌ Error al procesar el chunk {i + 1}: {e}")
    return uri_to_duration


In [18]:
# Definimos funciones auxiliares que buscan en el diccionario por cada fila
uri_to_duration_data = get_songs_durations(data)
uri_to_duration_sub = get_songs_durations(submission)
# Unimos los dos diccionarios
diccionario = {**uri_to_duration_data, **uri_to_duration_sub}

def get_duration(uri):
    return diccionario.get(uri, None)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98


In [19]:
# Creamos las nuevas columnas aplicando la funcion
train_data['duration_ms'] = train_data['spotify_track_uri'].apply(get_duration)
val_data['duration_ms'] = val_data['spotify_track_uri'].apply(get_duration)
submission['duration_ms'] = submission['spotify_track_uri'].apply(get_duration)

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.preprocessing import PolynomialFeatures

In [21]:
# Calculamos las proporciones para cada canción, artista y álbum
train_data['track_prop'] = train_data.groupby('master_metadata_track_name')['master_metadata_track_name'].transform('count')/len(train_data)
train_data['artist_prop'] = train_data.groupby('master_metadata_album_artist_name')['master_metadata_album_artist_name'].transform('count')/len(train_data)
train_data['album_prop'] = train_data.groupby('master_metadata_album_album_name')['master_metadata_album_album_name'].transform('count')/len(train_data)

val_data['track_prop'] = val_data.groupby('master_metadata_track_name')['master_metadata_track_name'].transform('count')/len(val_data)
val_data['artist_prop'] = val_data.groupby('master_metadata_album_artist_name')['master_metadata_album_artist_name'].transform('count')/len(val_data)
val_data['album_prop'] = val_data.groupby('master_metadata_album_album_name')['master_metadata_album_album_name'].transform('count')/len(val_data)

submission['track_prop'] = submission.groupby('master_metadata_track_name')['master_metadata_track_name'].transform('count')/len(submission)
submission['artist_prop'] = submission.groupby('master_metadata_album_artist_name')['master_metadata_album_artist_name'].transform('count')/len(submission)
submission['album_prop'] = submission.groupby('master_metadata_album_album_name')['master_metadata_album_album_name'].transform('count')/len(submission)

In [22]:
train_data

,Unnamed: 0,ts,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,reason_start,shuffle,TARGET,hour,...,month,year,is_iphone,reason_fwdbtn,reason_trackdone,reason_other,duration_ms,track_prop,artist_prop,album_prop
0,74917,2014-06-27 18:01:16+00:00,Algo Demencial,Los Tipitos,Push,spotify:track:0rqD0zvqI4GrvyUxyQ7Ij3,fwdbtn,True,True,18,...,6,2014,1,1,0,0,344466.0,0.000012,0.002135,0.000687
1,74918,2014-06-29 20:10:10+00:00,Iron Man - Live,Black Sabbath,Past Lives,spotify:track:1XtZ3GROvmy4JrT6uMvsD8,other,False,True,20,...,6,2014,1,0,0,1,385880.0,0.000025,0.000487,0.000025
2,74919,2014-06-29 20:10:54+00:00,Sultans Of Swing,Dire Straits,Dire Straits,spotify:track:3LTMnFa0hhwisyq6ILahyj,fwdbtn,False,True,20,...,6,2014,1,1,0,0,348624.0,0.000374,0.000911,0.000275
3,74920,2014-06-29 20:11:16+00:00,Fortunate Son,Creedence Clearwater Revival,Willy And The Poor Boys,spotify:track:7I2lPuuiOkpKtZlhr4zUpT,fwdbtn,False,True,20,...,6,2014,1,1,0,0,140773.0,0.000412,0.001361,0.000699
4,74921,2014-09-04 21:46:24+00:00,"Comptine d'un autre été, l'après-midi",Yann Tiersen,Amelie from Montmartre,spotify:track:2AkcjsKlRbIBYGAgpQVFii,other,False,True,21,...,9,2014,1,0,0,1,140733.0,0.000025,0.000025,0.000025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80105,74211,2024-04-23 18:10:42+00:00,Redbone,Childish Gambino,"""Awaken, My Love!""",spotify:track:0WtDGnWL2KrMCk0mI1Gpwz,trackdone,True,False,18,...,4,2024,1,0,1,0,326933.0,0.002259,0.012208,0.005530
80106,74212,2024-04-23 18:11:29+00:00,Thunder,Imagine Dragons,Evolve,spotify:track:1zB4vmk8tFRmM9UULNzbLB,trackdone,True,True,18,...,4,2024,1,0,1,0,187146.0,0.000836,0.008364,0.002933
80107,74213,2024-04-23 18:11:31+00:00,Pump Up The Jam - Edit,Technotronic,Best Of,spotify:track:0UAEHlFR79k9CJvknSGUNf,fwdbtn,True,True,18,...,4,2024,1,1,0,0,215040.0,0.001086,0.001086,0.001098
80108,74214,2024-04-23 18:11:33+00:00,Young Folks,Peter Bjorn and John,Writer's Block,spotify:track:4dyx5SzxPPaD8xQIid5Wjj,fwdbtn,True,True,18,...,4,2024,1,1,0,0,276693.0,0.001348,0.001373,0.001361


In [23]:
# Elegimos las columnas a usar para la primera combinación polinómica
def comb_polinom(df, cols):
    # Crear combinaciones polinómicas de las columnas especificadas
    poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
    poly_features = poly.fit_transform(df[cols])
    
    # Crear un DataFrame con los nombres de las nuevas columnas
    feature_names = poly.get_feature_names_out(cols)
    poly_df = pd.DataFrame(poly_features, columns=feature_names, index=df.index)
    poly_df = poly_df.drop(columns=cols)  # Eliminar las columnas originales
    
    # Concatenar el DataFrame original con el nuevo DataFrame de características polinómicas
    return pd.concat([df.reset_index(drop=True), poly_df.reset_index(drop=True)], axis=1)

In [24]:
train_data = comb_polinom(train_data, ['hour', 'day_of_week'])
val_data = comb_polinom(val_data, ['hour', 'day_of_week'])
submission = comb_polinom(submission, ['hour', 'day_of_week'])

In [25]:
#skips seguidos

def calculate_skips_seguidos(df):
    # Inicializamos la columna en 0
    df['skips_seguidos'] = 0
    
    # Iteramos desde la segunda fila
    for i in range(1, len(df)):
        try:
            ts_anterior = df.at[i-1, 'ts']
            duracion_anterior = pd.Timedelta(milliseconds=df.at[i-1, 'duration_ms'])
            ts_actual = df.at[i, 'ts']
            
            if ts_anterior + duracion_anterior > ts_actual:
                df.at[i, 'skips_seguidos'] = df.at[i-1, 'skips_seguidos'] + 1
        except:
            # Si hay un error, lo ignoramos y continuamos
            pass

calculate_skips_seguidos(train_data)
calculate_skips_seguidos(val_data)
calculate_skips_seguidos(submission)

In [26]:
# Entrenamiento y evaluación del modelo Random Forest
N_TREES = 500

rf = RandomForestClassifier(n_estimators=N_TREES, n_jobs=-1, random_state=6789, verbose=1, oob_score=True)
rf.fit(train_data.drop(columns=['ts','TARGET', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'reason_start']), train_data['TARGET'])

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:    2.9s
[Parallel(n_jobs=-1)]: Done 192 tasks      | elapsed:   13.9s
[Parallel(n_jobs=-1)]: Done 442 tasks      | elapsed:   35.0s
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:   40.1s finished


RandomForestClassifier(n_estimators=500, n_jobs=-1, oob_score=True,
                       random_state=6789, verbose=1)

In [ ]:
# Predecimos sobre el conjunto de validación
prediccion_val = rf.predict_proba(val_data.drop(columns=['ts','master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'reason_start', 'TARGET']))[:, 1]

auc = roc_auc_score(val_data['TARGET'], prediccion_val)
print(f"AUC-ROC: {auc:.4f}")

In [ ]:
prediccion_submmit = rf.predict_proba(submission.drop(columns=['ts','master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'reason_start']))[:, 1]

In [34]:
results = pd.DataFrame({'ID': submission['Unnamed: 0'], 'TARGET': prediccion_submmit})
results = results.set_index('ID').loc[submission_aux['Unnamed: 0']].reset_index()
results.to_csv('submission_9.csv', index=False)